# 02 — Model training and evaluation (reproduces the R08 noisy-OR run)

This notebook reproduces the **compound-holdout** results for the R08-augmented v8
honest-exposure cohort (mean-of-folds AUC, 5 seeds × 5 folds StratifiedGroupKFold grouped by
compound SMILES). It asserts the reproduced numbers against the committed metrics in
`results/production_v8_honest_exposure_noisyor_R08/metrics.json` (the exact headline is read from
that file, not restated here).

It **drives the production functions** in `scripts/retrain_calibrated.py` directly (no
re-implementation), so the notebook cannot diverge from the released pipeline. It is the
notebook equivalent of:

```
python scripts/retrain_calibrated.py --calibrate none --skip-crosstask \
    --safety-head noisy_or \
    --data data/sources/training_dataset_v8_honest_exposure.csv \
    --out results/production_v8_honest_exposure_noisyor_R08
```

**Provenance:** see `results/production_v8_honest_exposure_noisyor_R08/metrics.json.provenance.json`
(data sha256, seeds `[42, 123, 456, 789, 2024]`, pipeline `retrain_calibrated.py`).

**Model.** Gradient-boosted decision trees. The *overall* task and each per-mechanism *safety*
detector use a single scikit-learn GBM; the *efficacy* task uses an equal-weight average of
GBM + XGBoost + LightGBM (balanced class weights); the *safety* task in this run is a
parameter-free noisy-OR over six per-mechanism detectors (the decomposition is retained for its
disjunctive inductive bias). Nested top-20 feature selection and median imputation are fit within
each training fold (no leakage). Pipeline Step 1 (raw AACT → features, incl. the R08 recovery +
combination-attribution v2 + dual label audits) is in `01_data_provenance_rebuild_executed.ipynb`;
this is Step 2 (features → model → metrics).

> **Note:** cached outputs below may predate the R08 regeneration; re-executing the notebook
> regenerates the R08 metrics and the §4 assertion compares against the committed `_R08` metrics.

In [ ]:
import json, sys
import numpy as np, pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
import retrain_calibrated as rc   # production training/eval functions

DATA      = ROOT / 'data/sources/training_dataset_v8_honest_exposure.csv'
COMMITTED = ROOT / 'results/production_v8_honest_exposure_noisyor_R08/metrics.json'  # R08-augmented cohort
print('production functions imported from', rc.__file__)
print('seeds (rc.SEEDS):', rc.SEEDS)

## 1. Load the training dataset and candidate features

In [2]:
df = pd.read_csv(DATA, low_memory=False)
feature_cols = rc.get_features(df)   # NB: called on the raw frame before any _y/outcome col is added
print(f'Trials = {len(df)}   Drugs (unique SMILES) = {df.SMILES.nunique()}   '
      f'Candidate features = {len(feature_cols)}')
df['Corrected_Outcome'].value_counts()

Trials = 3182   Drugs (unique SMILES) = 773   Candidate features = 219


Corrected_Outcome
PASS                    2803
FAIL_EFFICACY            273
FAIL_SAFETY               86
EXCLUDE_NONDRUG_STOP      12
FAIL_BOTH                  8
Name: count, dtype: int64

## 2. Build the per-task cohorts

Mirrors `retrain_calibrated.main()` exactly: efficacy excludes anti-pathogen / endogenous /
supportive / healthy-volunteer / procedural / multi-drug rows; safety excludes multi-drug rows;
the label `_y` is the per-task failure indicator.

In [3]:
safety_mask = df['Corrected_Outcome'].isin(['PASS', 'FAIL_SAFETY', 'FAIL_BOTH'])
if 'is_multi_drug_exclude' in df.columns:
    safety_mask &= df['is_multi_drug_exclude'] != 1
df_safety = df[safety_mask].copy()
df_safety['_y'] = df_safety['Corrected_Outcome'].isin(['FAIL_SAFETY', 'FAIL_BOTH']).astype(int)

eff_excl = pd.Series(False, index=df.index)
for c in ['is_anti_pathogen', 'is_endogenous', 'is_mispaired_supportive',
          'is_healthy_volunteer', 'is_procedural_exclude', 'is_multi_drug_exclude']:
    if c in df.columns:
        eff_excl |= df[c] == 1
df_eff = df[~eff_excl & df['Corrected_Outcome'].isin(['PASS', 'FAIL_EFFICACY', 'FAIL_BOTH'])].copy()
df_eff['_y'] = df_eff['Corrected_Outcome'].isin(['FAIL_EFFICACY', 'FAIL_BOTH']).astype(int)

df_over = df[df['Corrected_Outcome'].isin(['PASS', 'FAIL_SAFETY', 'FAIL_EFFICACY', 'FAIL_BOTH'])].copy()
df_over['_y'] = df_over['Corrected_Outcome'].isin(['FAIL_SAFETY', 'FAIL_EFFICACY', 'FAIL_BOTH']).astype(int)

for name, d in [('overall', df_over), ('safety', df_safety), ('efficacy', df_eff)]:
    print(f'{name:9s} n={len(d):5d}  positives={int(d._y.sum()):4d}  '
          f'pos_rate={d._y.mean():.3f}  drugs={d.SMILES.nunique()}')

overall   n= 3170  positives= 367  pos_rate=0.116  drugs=769
safety    n= 2894  positives=  94  pos_rate=0.032  drugs=639
efficacy  n= 2575  positives= 257  pos_rate=0.100  drugs=644


## 3. Run the three task heads under compound-holdout CV

This is the compute-heavy cell (5 seeds × 5 folds × {1 GBM overall, 3-model ensemble efficacy,
6 detectors × noisy-OR safety}). `calibrate=False` matches the `--calibrate none` production run
whose headline the paper reports.

In [4]:
safety_protect = [c for c in feature_cols if c == 'logdose' or c.endswith('_xdose')]

# Safety: noisy-OR over per-mechanism detectors (the published safety head)
oof_s, fm_s, detail_s = rc.noisy_or_safety_oof(
    df_safety, feature_cols, protect_cols=safety_protect, return_detail=True)

# Efficacy: GBM + XGBoost + LightGBM ensemble
oof_e, fm_e = rc.run_task_cv(df_eff, feature_cols, 'efficacy', calibrate=False)

# Overall: single GBM
oof_o, fm_o = rc.run_task_cv(df_over, feature_cols, 'overall', calibrate=False)
print('done')

  [safety] noisy-OR over 6 mechanism detectors: {'promiscuity': 58, 'hepatic_dili': 15, 'cardiac': 6, 'network': 27, 'tissue': 70, 'context': 43}


  [safety] protecting 6 features from selection: ['binding_drug_n_bound_xdose', 'binding_n_above_80_xdose', 'tox_antitarget_burden_xdose', 'tox_renal_burden_xdose', 'binding_score_max_xdose', 'tox_vs_overall_ratio_xdose']


  safety seed=42 fold=0 AUC_raw=0.788 AUC_cal=0.788


  safety seed=42 fold=1 AUC_raw=0.678 AUC_cal=0.678


  safety seed=42 fold=2 AUC_raw=0.590 AUC_cal=0.590


...

  overall seed=2024 fold=1 AUC_raw=0.824 AUC_cal=0.824


  overall seed=2024 fold=2 AUC_raw=0.765 AUC_cal=0.765


  overall seed=2024 fold=3 AUC_raw=0.730 AUC_cal=0.730


  overall seed=2024 fold=4 AUC_raw=0.787 AUC_cal=0.787


done


## 4. Headline mean-of-folds AUC — reproduced vs committed

In [ ]:
def mof(fm):
    return float(np.nanmean(fm['auc_raw'])), float(np.nanstd(fm['auc_raw']))

repro = {'overall': mof(fm_o), 'efficacy': mof(fm_e), 'safety': mof(fm_s)}
committed = json.load(open(COMMITTED))

print(f'{"task":9s} {"reproduced (mean ± SD)":>26s}   {"committed":>10s}   match')
ok = True
for t in ['overall', 'efficacy', 'safety']:
    r, sd = repro[t]
    c = committed[t]['mean_of_folds']['auc_raw']
    m = abs(r - c) < 1e-4
    ok &= m
    print(f'{t:9s}   {r:.6f} ± {sd:.3f}        {c:.6f}     {"OK" if m else "MISMATCH"}')

assert ok, 'Reproduction mismatch vs committed metrics.json!'
print('\nReproduces the published headline (mean-of-folds AUC): '
      'overall 0.797 / efficacy 0.796 / safety 0.728')

## 5. Scope and the rest of the analysis (full R08 reproduction recipe)

This notebook reproduces the **headline compound-holdout prediction** (manuscript Results,
Fig. 2, Table 1, Supplementary Table S1). Every other manuscript analysis is a dedicated,
deterministic script driven by the same R08 cohort (`training_dataset_v8_honest_exposure.csv`)
and seeds. Run order to regenerate the full v8/R08 manuscript:

**Step 0 — data (run once):** `01_data_provenance_rebuild_executed.ipynb` → `training_dataset_v5_unified.csv`;
then `build_v8_dataset.py` → merge `combination_attribution_v2.csv` filtered to `attr_misindexed==False`
→ `training_dataset_v8_honest.csv`; `build_v8_honest_exposure.py`; `apply_safety_label_corrections.py`
(glob-applies all `*_label_corrections_*.csv`, incl. the R08 safety+efficacy audits).

| Analysis (manuscript location) | Script | Output |
|---|---|---|
| Headline trial-level (Fig 2a, Table 1, Supp Table S1) | `retrain_calibrated.py --calibrate none --skip-crosstask --safety-head noisy_or` | `results/production_v8_honest_exposure_noisyor_R08/` |
| Single-head safety comparator (noisy-OR demotion) | `retrain_calibrated.py … --safety-head single` | `…_single_R08/` |
| Arm-level (Supp Table S14) | `retrain_arm_level_v18_production.py` | `results/arm_level_v18/` |
| Signal decomposition (Fig 2c, Supp Table S1) | `decompose_v8_noclass.py` | stdout |
| Phase-matched efficacy (Methods) | `phase_stratified_auc.py` | stdout |
| Temporal leave-future-out (Supp Table S4) | `temporal_leave_future_out.py` | `results/temporal_v8/leave_future_out.json` |
| Temporal feature ablation (Supp Table S4) | `temporal_feature_ablation.py` | stdout |
| Public-only / causal-plausibility decomposition | `decompose_v8_public_cleanmort.py` | stdout |
| Cross-task triage + noisy-OR (Supp Table S12) | `retrain_calibrated.py --calibrate isotonic --safety-head noisy_or --out …_crosstask_R08`; `noisy_or_crosstask_safety.py` | `…_crosstask_R08/` |
| External safety-axis validation (hERG/SIDER/DILIrank, Supp Table S3) | `safety_detector_external_validation.py`, `cardiac_axis_hardening.py` | `results/` |
| Repositioning / rescue (Fig 4, Supp Table S7) | `within_drug_genetic_contrast.py`, `results/rescue_*.csv` | — |
| Nomination real-world grounding (S9) | workflow `nomination-grounding` (web-verify) | `results/rescue_nomination_grounding.csv` |
| Figures | `make_figures_v8.py`, `phase1/make_fig{1,3}_v8.py`, `phase1/compose_figures.py` | `manuscript/figures_v8/` |
| Manuscript exports | `build_manuscript_docx.py`; `build_manuscript_latex.py`; `build_supplementary.py` | `manuscript/*.pdf/.docx/.html` |

Because §3 calls the production functions in `retrain_calibrated.py` directly, the notebook stays
in lock-step with the released pipeline: re-running it regenerates the exact committed R08 metrics
(asserted in §4). `03_supporting_analyses.ipynb` covers the PCA/SIDER/calibration supporting notes.